In [1]:
from pathlib import Path

def get_images_from_directory(directory_path):
    image_extensions = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp'}
    path = Path(directory_path)
    
    images = []
    for file_path in path.iterdir():
        if file_path.is_file() and file_path.suffix.lower() in image_extensions:
            images.append(str(file_path))
    
    return images

# Usage
af_folder = "/home/dnth/Downloads/bone-break/Bone Break Classification/Bone Break Classification/Avulsion fracture/Train"
cf_folder = "/home/dnth/Downloads/bone-break/Bone Break Classification/Bone Break Classification/Comminuted fracture/Train"

cat_image_list = get_images_from_directory(af_folder)
dog_image_list = get_images_from_directory(cf_folder)

In [2]:
NUM_SAMPLES = 8
image_paths = cat_image_list[:NUM_SAMPLES] + dog_image_list[:NUM_SAMPLES]
image_paths

['/home/dnth/Downloads/bone-break/Bone Break Classification/Bone Break Classification/Avulsion fracture/Train/jcs372-g001_jpg.rf.c7a28478b9626ae41e243f25c2618e95.jpg',
 '/home/dnth/Downloads/bone-break/Bone Break Classification/Bone Break Classification/Avulsion fracture/Train/Segond-Fx-label-2_png.rf.ebecbcb0ff895e7e06cb974569469.jpg',
 '/home/dnth/Downloads/bone-break/Bone Break Classification/Bone Break Classification/Avulsion fracture/Train/images_jpg.rf.0ce1d5a7ca4c319ff39d4445df37f3f.jpg',
 '/home/dnth/Downloads/bone-break/Bone Break Classification/Bone Break Classification/Avulsion fracture/Train/image13_jpeg.rf.629135763282eca324d34d3bdde57075.jpg',
 '/home/dnth/Downloads/bone-break/Bone Break Classification/Bone Break Classification/Avulsion fracture/Train/image11_jpeg.rf.4da4acf6d2c1d061687b597494c6ee8c.jpg',
 '/home/dnth/Downloads/bone-break/Bone Break Classification/Bone Break Classification/Avulsion fracture/Train/image9_jpeg.rf.4f7d7cb5a7a3ff33253bb4769fdaf32c.jpg',
 '/ho

In [3]:
from setfit import SetFitImageModel, SetFitImageTrainer, TrainingArguments

model = SetFitImageModel(
    timm_model_name="timm/resnet50.a1_in1k",
    train_embeddings=True,
    labels=["af", "cf"]
)

# Train with image paths and labels
trainer = SetFitImageTrainer(model=model)

args = TrainingArguments(
    batch_size=4,
    num_epochs=(3, 30),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer.train(x_train=image_paths, y_train=['af'] * NUM_SAMPLES + ['cf'] * NUM_SAMPLES, args=args)
# trainer.train(x_train=image_paths, y_train=['af'] * NUM_SAMPLES + ['cf'] * NUM_SAMPLES)

Training TIMM model embeddings for image model


Epoch:   0%|          | 0/3 [00:00<?, ?it/s]

Iteration:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 1/3, Loss: 1.1454


Iteration:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 2/3, Loss: 1.4392


Iteration:   0%|          | 0/4 [00:00<?, ?it/s]

Epoch 3/3, Loss: 1.2239


In [4]:
model.predict("/home/dnth/Downloads/bone-break/Bone Break Classification/Bone Break Classification/Comminuted fracture/Test/images94_jpg.rf.1be75680f0289a4d2bcc8d0c112a3453.jpg", show_progress_bar=True)

Encoding images:   0%|          | 0/1 [00:00<?, ?batch/s]

'af'

In [5]:
model.predict_proba("/home/dnth/Downloads/bone-break/Bone Break Classification/Bone Break Classification/Comminuted fracture/Test/images94_jpg.rf.1be75680f0289a4d2bcc8d0c112a3453.jpg", show_progress_bar=True)

Encoding images:   0%|          | 0/1 [00:00<?, ?batch/s]

tensor([0.5525, 0.4475], dtype=torch.float64)

## Eval on Cat Dogs dataset

In [6]:
import pandas as pd

image_list = cat_image_list + dog_image_list

df = pd.DataFrame(image_list, columns=['image_path'])
df


,image_path
0,/home/dnth/Downloads/bone-break/Bone Break Cla...
1,/home/dnth/Downloads/bone-break/Bone Break Cla...
2,/home/dnth/Downloads/bone-break/Bone Break Cla...
3,/home/dnth/Downloads/bone-break/Bone Break Cla...
4,/home/dnth/Downloads/bone-break/Bone Break Cla...
...,...
238,/home/dnth/Downloads/bone-break/Bone Break Cla...
239,/home/dnth/Downloads/bone-break/Bone Break Cla...
240,/home/dnth/Downloads/bone-break/Bone Break Cla...
241,/home/dnth/Downloads/bone-break/Bone Break Cla...


In [7]:
df['label'] = df['image_path'].str.split('/').str[-3].str.lower()
df['label'] = df['label'].replace('avulsion fracture', 'af')
df['label'] = df['label'].replace('comminuted fracture', 'cf')

In [8]:
df

,image_path,label
0,/home/dnth/Downloads/bone-break/Bone Break Cla...,af
1,/home/dnth/Downloads/bone-break/Bone Break Cla...,af
2,/home/dnth/Downloads/bone-break/Bone Break Cla...,af
3,/home/dnth/Downloads/bone-break/Bone Break Cla...,af
4,/home/dnth/Downloads/bone-break/Bone Break Cla...,af
...,...,...
238,/home/dnth/Downloads/bone-break/Bone Break Cla...,cf
239,/home/dnth/Downloads/bone-break/Bone Break Cla...,cf
240,/home/dnth/Downloads/bone-break/Bone Break Cla...,cf
241,/home/dnth/Downloads/bone-break/Bone Break Cla...,cf


In [9]:
# df = df.sample(1000)

In [10]:
# Run batch inference
df['pred'] = model.predict(df['image_path'].tolist(), batch_size=64, show_progress_bar=True)

Encoding images:   0%|          | 0/4 [00:00<?, ?batch/s]

In [11]:
df

,image_path,label,pred
0,/home/dnth/Downloads/bone-break/Bone Break Cla...,af,af
1,/home/dnth/Downloads/bone-break/Bone Break Cla...,af,af
2,/home/dnth/Downloads/bone-break/Bone Break Cla...,af,af
3,/home/dnth/Downloads/bone-break/Bone Break Cla...,af,af
4,/home/dnth/Downloads/bone-break/Bone Break Cla...,af,af
...,...,...,...
238,/home/dnth/Downloads/bone-break/Bone Break Cla...,cf,af
239,/home/dnth/Downloads/bone-break/Bone Break Cla...,cf,cf
240,/home/dnth/Downloads/bone-break/Bone Break Cla...,cf,cf
241,/home/dnth/Downloads/bone-break/Bone Break Cla...,cf,cf


In [12]:
accuracy = (df['label'] == df['pred']).mean()
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.6296


In [13]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Accuracy (same as above)
acc = accuracy_score(df['label'], df['pred'])
print(f"Accuracy: {acc:.4f}\n")

# Detailed classification report
print("Classification Report:")
print(classification_report(df['label'], df['pred']))

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(df['label'], df['pred']))

Accuracy: 0.6296

Classification Report:
              precision    recall  f1-score   support

          af       0.56      0.83      0.67       109
          cf       0.77      0.47      0.58       134

    accuracy                           0.63       243
   macro avg       0.66      0.65      0.62       243
weighted avg       0.67      0.63      0.62       243

Confusion Matrix:
[[90 19]
 [71 63]]
